# Scientific Paper Knowledge Graph with Embeddings and Local LLMs

Course project pipeline for building a domain-oriented knowledge graph from scientific PDF papers.

Final graphs:

1. Embedding Similarity Graph
2. LLM Relation Graph
3. Weighted Hybrid Knowledge Graph


In [ ]:
# 1. Install dependencies
# In Google Colab, run this cell if packages are missing.
# !pip install -q pandas numpy PyMuPDF sentence-transformers scikit-learn networkx pyvis matplotlib transformers torch accelerate tqdm


In [ ]:
# 2. Configuration
import os
from pathlib import Path
import sys

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PAPERS_PATH = PROJECT_ROOT / "data" / "papers"
RESULTS_PATH = PROJECT_ROOT / "data" / "results"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SIMILARITY_THRESHOLD = 0.55
TOP_K = 5
LLM_SIMILARITY_THRESHOLD = 0.50
LLM_TOP_K = 10
LLM_MAX_PAIRS = None
USE_LLM = True
HYBRID_ALPHA = 0.5

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Papers path:", PAPERS_PATH)
print("Results path:", RESULTS_PATH)


In [ ]:
# 3. Imports
import networkx as nx
import pandas as pd
from tqdm.auto import tqdm

from src.pdf_parser import parse_papers_from_folder
from src.preprocessing import clean_text, truncate_text
from src.embeddings import (
    load_embedding_model,
    compute_embeddings,
    compute_similarity_matrix,
    get_candidate_pairs,
)
from src.llm_relation_extractor import load_llm, classify_relation
from src.graph_builder import (
    build_similarity_graph,
    filter_llm_relations,
    build_llm_relation_graph,
    build_weighted_hybrid_graph,
    graph_to_nodes_df,
    graph_to_edges_df,
)
from src.graph_metrics import (
    compare_graphs,
    get_communities_df,
    get_community_summary_df,
    calculate_centrality_df,
    explain_path,
    find_paper_id_by_filename,
)
from src.visualization import visualize_graph_pyvis, plot_metric_comparison
from src.utils import save_dataframe, save_graph_graphml


In [ ]:
# 4-6. Parse PDFs, clean text, and optionally fix titles
papers_df = parse_papers_from_folder(PAPERS_PATH)

TITLE_MAP = {
    # "qlora.pdf": "QLoRA: Efficient Finetuning of Quantized LLMs",
    # "BERT.pdf": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
}
if not papers_df.empty:
    papers_df["title"] = papers_df.apply(
        lambda row: TITLE_MAP.get(str(row["filename"]), row["title"]),
        axis=1,
    )
    papers_df["text_clean"] = papers_df["text"].fillna("").apply(clean_text)
    papers_df["text_for_embedding"] = papers_df.apply(
        lambda row: truncate_text(
            row["abstract"] if isinstance(row["abstract"], str) and row["abstract"].strip() else row["text_clean"],
            5000,
        ),
        axis=1,
    )
else:
    papers_df["text_clean"] = []
    papers_df["text_for_embedding"] = []

save_dataframe(papers_df, RESULTS_PATH / "papers.csv")
print(f"Parsed papers: {len(papers_df)}")
display(papers_df[["paper_id", "filename", "title", "abstract"]].head(20))


In [ ]:
# 7-9. Embeddings, similarity matrix, and candidate pairs
embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)
embeddings = compute_embeddings(papers_df["text_for_embedding"].tolist(), embedding_model)
similarity_matrix = compute_similarity_matrix(embeddings)

candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=SIMILARITY_THRESHOLD, top_k=TOP_K)
llm_candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=LLM_SIMILARITY_THRESHOLD, top_k=LLM_TOP_K)

candidate_pairs_df = pd.DataFrame(candidate_pairs, columns=["source", "target", "similarity"])
save_dataframe(candidate_pairs_df, RESULTS_PATH / "candidate_pairs.csv")

print(f"Embedding graph candidate pairs: {len(candidate_pairs)}")
print(f"LLM candidate pairs: {len(llm_candidate_pairs)}")
display(candidate_pairs_df.head())


In [ ]:
# 10. Embedding Similarity Graph
embedding_graph = build_similarity_graph(papers_df, candidate_pairs)
save_graph_graphml(embedding_graph, RESULTS_PATH / "embedding_graph.graphml")
print("Embedding Similarity Graph:", embedding_graph)


In [ ]:
# 11-13. Load local LLM and extract typed relations
LLM_COLUMNS = [
    "source",
    "target",
    "relation",
    "confidence",
    "reason",
    "candidate_similarity",
    "source_method",
]
llm_relations_raw = []

if USE_LLM and llm_candidate_pairs:
    llm_model, tokenizer = load_llm(LLM_MODEL_NAME)
    papers_by_id = papers_df.set_index("paper_id").to_dict(orient="index")
    pairs_for_llm = llm_candidate_pairs[:LLM_MAX_PAIRS] if LLM_MAX_PAIRS else llm_candidate_pairs

    for pair in tqdm(pairs_for_llm, desc="LLM relation extraction"):
        paper_a = {"paper_id": int(pair["source"]), **papers_by_id[int(pair["source"])]}
        paper_b = {"paper_id": int(pair["target"]), **papers_by_id[int(pair["target"])]}
        result = classify_relation(paper_a, paper_b, llm_model, tokenizer)
        result["candidate_similarity"] = float(pair["similarity"])
        result["source_method"] = "llm"
        llm_relations_raw.append(result)
else:
    print("Local LLM extraction is disabled or there are no candidate pairs.")

llm_relations_raw_df = pd.DataFrame(llm_relations_raw)
for column in LLM_COLUMNS:
    if column not in llm_relations_raw_df.columns:
        llm_relations_raw_df[column] = ""
llm_relations_raw_df = llm_relations_raw_df[LLM_COLUMNS]
save_dataframe(llm_relations_raw_df, RESULTS_PATH / "llm_relations_raw.csv")

print(f"Raw LLM relation rows: {len(llm_relations_raw_df)}")
display(llm_relations_raw_df.head())


In [ ]:
# 14-16. Filter LLM edges and build LLM Relation Graph
llm_relations_filtered_df = filter_llm_relations(llm_relations_raw_df)
save_dataframe(llm_relations_filtered_df, RESULTS_PATH / "llm_relations_filtered.csv")

llm_graph = build_llm_relation_graph(papers_df, llm_relations_filtered_df)
save_graph_graphml(llm_graph, RESULTS_PATH / "llm_graph.graphml")

print(f"Filtered LLM relation rows: {len(llm_relations_filtered_df)}")
print("LLM Relation Graph:", llm_graph)
display(llm_relations_filtered_df.head())


In [ ]:
# 17. Weighted Hybrid Knowledge Graph
if USE_LLM:
    hybrid_graph = build_weighted_hybrid_graph(
        papers_df,
        embedding_graph,
        llm_relations_filtered_df,
        alpha=HYBRID_ALPHA,
    )
else:
    hybrid_graph = embedding_graph.copy()

save_graph_graphml(hybrid_graph, RESULTS_PATH / "hybrid_graph.graphml")
save_dataframe(graph_to_nodes_df(hybrid_graph), RESULTS_PATH / "hybrid_nodes.csv")
save_dataframe(graph_to_edges_df(hybrid_graph), RESULTS_PATH / "hybrid_edges.csv")

print("Weighted Hybrid Knowledge Graph:", hybrid_graph)
display(graph_to_edges_df(hybrid_graph).head())


In [ ]:
# 18. Graph metrics
if USE_LLM:
    graphs = {
        "Embedding Similarity Graph": embedding_graph,
        "LLM Relation Graph": llm_graph,
        "Weighted Hybrid Knowledge Graph": hybrid_graph,
    }
else:
    graphs = {
        "Embedding Similarity Graph": embedding_graph,
        "Weighted Hybrid Knowledge Graph": hybrid_graph,
    }

metrics_df = compare_graphs(graphs)
save_dataframe(metrics_df, RESULTS_PATH / "graph_metrics.csv")

display(metrics_df)


In [ ]:
# 19. Communities for the hybrid graph
communities_df = get_communities_df(hybrid_graph)
community_summary_df = get_community_summary_df(communities_df)

save_dataframe(communities_df, RESULTS_PATH / "communities.csv")
save_dataframe(community_summary_df, RESULTS_PATH / "community_summary.csv")

print(f"Communities: {community_summary_df.shape[0]}")
display(community_summary_df)


In [ ]:
# 20. Centrality for the hybrid graph
centrality_df = calculate_centrality_df(hybrid_graph)
save_dataframe(centrality_df, RESULTS_PATH / "centrality.csv")

display(centrality_df.head(10))


In [ ]:
# 21. Shortest path explanation example
path_example_df = pd.DataFrame()
path_candidates = [
    ("qlora", "BERT"),
    ("selfrag", "rag"),
]

for source_part, target_part in path_candidates:
    try:
        source_id = find_paper_id_by_filename(papers_df, source_part)
        target_id = find_paper_id_by_filename(papers_df, target_part)
        path_example_df = explain_path(hybrid_graph, source_id, target_id)
        if not path_example_df.empty:
            print(f"Path example: {source_part} -> {target_part}")
            break
    except ValueError as exc:
        print(exc)

if path_example_df.empty and hybrid_graph.number_of_edges() > 0:
    component = max(nx.connected_components(hybrid_graph), key=len)
    nodes = list(component)
    if len(nodes) >= 2:
        path_example_df = explain_path(hybrid_graph, int(nodes[0]), int(nodes[-1]))
        print(f"Fallback path example: {nodes[0]} -> {nodes[-1]}")

save_dataframe(path_example_df, RESULTS_PATH / "path_example.csv")
display(path_example_df)


In [ ]:
# 22. Optional visualization
try:
    visualize_graph_pyvis(embedding_graph, str(RESULTS_PATH / "embedding_similarity_graph.html"))
    if USE_LLM:
        visualize_graph_pyvis(llm_graph, str(RESULTS_PATH / "llm_relation_graph.html"))
    visualize_graph_pyvis(hybrid_graph, str(RESULTS_PATH / "hybrid_graph.html"))

    for metric_name in ["number_of_edges", "density", "average_degree", "average_clustering", "number_of_communities"]:
        plot_metric_comparison(metrics_df, metric_name, str(RESULTS_PATH / f"compare_{metric_name}.png"))
    print("Visualizations saved to", RESULTS_PATH)
except Exception as exc:
    print("Visualization skipped:", exc)


In [ ]:
# 23. Export summary
expected_files = [
    "papers.csv",
    "candidate_pairs.csv",
    "llm_relations_raw.csv",
    "llm_relations_filtered.csv",
    "graph_metrics.csv",
    "communities.csv",
    "community_summary.csv",
    "centrality.csv",
    "path_example.csv",
    "hybrid_nodes.csv",
    "hybrid_edges.csv",
    "embedding_graph.graphml",
    "llm_graph.graphml",
    "hybrid_graph.graphml",
]

for file_name in expected_files:
    path = RESULTS_PATH / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'missing'}")


## Interpretation

Embedding Graph gives broad coverage and connectivity.

LLM Relation Graph gives typed, interpretable semantic links.

Weighted Hybrid Knowledge Graph combines both approaches and is the final domain-oriented knowledge graph for analysis and defense.
